# Public demonstration: patient-specific directed transitions

This notebook demonstrates the public code using **synthetic data only**. It does not reproduce the private study cohort or the manuscript results.

For authorized local analysis, replace the synthetic input with a compatible institutional dataset and keep the patient-level data outside Git/version control.

In [ ]:
from pathlib import Path
import pandas as pd

from src.generate_synthetic_data import generate_synthetic_data, validate_synthetic_data
from src.pipeline import JAMIAFinalPipeline, TARGET_COLUMNS
from src.evaluation import run_statistical_evaluation


## 1. Generate or load the synthetic demonstration dataset

The public dataset includes a small synthetic CSV so the repository can be tested without protected healthcare data. The synthetic insurance categories include **Organization** (بیمه سازمانی).

In [ ]:
data_path = Path('data/synthetic_demo.csv')
if not data_path.exists():
    data_path = Path('../data/synthetic_demo.csv')
if not data_path.exists():
    df = generate_synthetic_data(n_patients=300, max_episodes=5, random_state=42)
    validate_synthetic_data(df)
    data_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(data_path, index=False)
else:
    df = pd.read_csv(data_path, parse_dates=['EpisodeStartDate', 'EpisodeEndDate'])

print(df.shape)
display(df.head())
print('Patients:', df['NationalCode'].nunique())
print('Episodes:', len(df))
print('Insurance:', sorted(df['InsuranceType'].unique()))


## 2. Build the patient-level split and historical features

The pipeline hashes the patient grouping identifier internally, uses an 80/20 patient-level split, defines Q75 targets from training data only, and constructs historical directed transitions from prior episodes.

In [ ]:
pipeline = JAMIAFinalPipeline(df, model_dir='models_demo', random_state=42)
pipeline.split_data()
pipeline.apply_binary_targets()
pipeline.build_features()

feature_sets = pipeline.get_feature_sets()
for name, features in feature_sets.items():
    print(name, len(features), 'features')


## 3. Fast smoke-test training

The public demo uses `n_iter=2` so that a new user can verify the repository quickly. The study analysis used the manuscript configuration with 30 randomized configurations.

In [ ]:
results = pipeline.run_all_targets(n_iter=2)

summary = []
for key, value in results.items():
    summary.append({
        'Model': key,
        'AUROC': value['AUROC'],
        'AUPRC': value['AUPRC'],
        'Brier': value['Brier'],
        'MCC@0.5': value['MCC'],
    })

pd.DataFrame(summary)


## 4. Statistical evaluation demonstration

This produces a synthetic-data demonstration of paired bootstrap comparisons and group-aware out-of-fold threshold selection. These numbers are **demo outputs**, not the reported manuscript results.

In [ ]:
demo_stats = run_statistical_evaluation(
    results=results,
    train_df=pipeline.train_df,
    test_df=pipeline.test_df,
    feature_sets=feature_sets,
    targets=list(TARGET_COLUMNS),
    bootstrap_iterations=50,
)

demo_stats


## 5. Authorized-data workflow

For a private, authorized dataset, load a compatible episode-level dataframe instead of `synthetic_demo.csv`. Do **not** commit the private CSV, hashed identifiers, trained models, or patient-level predictions to GitHub.